Accidentally reinvented the Stockham FFT after trying to do Cooley-Tukey in-place and without recursion.

In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmplyw4c1bi".


In [30]:
%%cuda

#include <stdio.h>
#include <cmath>
#include <complex>
#include <vector>
#include <cuda_runtime.h>
#include <cuda/std/complex>

constexpr size_t MAX_BLOCK_SIZE = 1024;
using complex = std::complex<double>;
using cu_complex = cuda::std::complex<double>;
constexpr double pi = 3.14159265358979323846;
cu_complex data[] = {
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 -1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0
};

__global__ void fft_cuda_kernel(cu_complex* data, size_t n) {
    size_t i{ blockIdx.x * blockDim.x + threadIdx.x };
    __syncthreads();

    for (size_t stride{n / 2}; stride > 0; stride /= 2) {
        __syncthreads();

        // note that j resets after i=n/2
        size_t j{ (i % (n/2)) / stride };
        size_t subprob_offset{ i % stride };
        cu_complex y_e_j;
        cu_complex y_o_j;
        y_e_j = data[2 * j * stride + subprob_offset];
        y_o_j = data[(2 * j + 1) * stride + subprob_offset];
        __syncthreads();
        double theta{ -2.0 * pi / (n / stride) * j };
        double real, imag;
        sincos(theta, &imag, &real);
        cu_complex omega_to_j{ real, imag };
        data[i] = y_e_j + (-1 + 2 * (static_cast<double>(i < n / 2))) * omega_to_j * y_o_j; // avoids branching. equivalent to (i < n / 2) ? 1 : -1
        __syncthreads();
    }
}

void fft_cuda(cu_complex* h_data, size_t n) {
    cu_complex* d_data;
    cudaMalloc((void**)&d_data, n * sizeof(cu_complex));
    cudaMemcpy(d_data, h_data, n * sizeof(cu_complex), cudaMemcpyHostToDevice);;

    fft_cuda_kernel<<<1, n>>>(d_data, n);

    cudaMemcpy(h_data, d_data, n * sizeof(cu_complex), cudaMemcpyDeviceToHost);
    cudaDeviceSynchronize();
    cudaFree(d_data);
}

int main() {
    size_t n{ sizeof(data) / sizeof(complex) };
    if (n > MAX_BLOCK_SIZE) return -1; // only works if everything fits on a block, for now

    fft_cuda(data, n);

    for (size_t i{}; i < n; i++) {
        printf("%d, %f", i, cuda::std::abs(data[i]));
        printf("\n");
    }
}

/////////////////////////////////////////////////////

void fft_cpu(complex* data, size_t n) {
    if (n == 1) {
        return;
    }

    complex omega{ std::exp(complex{0, - 2 * pi / n}) };
    std::vector<complex> data_e(n/2);
    std::vector<complex> data_o(n/2);
    for (size_t i{}; i < n/2; i++) {
        data_e[i] = data[2 * i];
        data_o[i] = data[2 * i + 1];
    }
    fft_cpu(data_e.data(), n/2);
    fft_cpu(data_o.data(), n/2);

    complex omega_to_i{ 1 };
    for (size_t i{}; i < n/2; i++) {
        data[i] = data_e[i] + omega_to_i * data_o[i];
        data[i + n/2] = data_e[i] - omega_to_i * data_o[i];
        omega_to_i *= omega;
    }
}

void ifft_cpu_recursion(complex* data, size_t n) {
    if (n == 1) {
        return;
    }

    complex omega{ std::exp(complex{0, 2 * pi / n}) };
    std::vector<complex> data_e(n/2);
    std::vector<complex> data_o(n/2);
    for (size_t i{}; i < n/2; i++) {
        data_e[i] = data[2 * i];
        data_o[i] = data[2 * i + 1];
    }
    ifft_cpu_recursion(data_e.data(), n/2);
    ifft_cpu_recursion(data_o.data(), n/2);

    complex omega_to_i{ 1 };
    for (size_t i{}; i < n/2; i++) {
        data[i] = (data_e[i] + omega_to_i * data_o[i]);
        data[i + n/2] = (data_e[i] - omega_to_i * data_o[i]) ;
        omega_to_i *= omega;
    }
}

void ifft_cpu(complex* data, size_t n) {
    ifft_cpu_recursion(data, n);
    double n_reciprocal{ 1 / static_cast<double>(n) };
    for (size_t i{}; i < n; i++) {
        data[i] *= n_reciprocal;
    }
}

0, 0.000000
1, 0.000000
2, 81.520065
3, 0.000000
4, 0.000000
5, 0.000000
6, 27.260867
7, 0.000000
8, 0.000000
9, 0.000000
10, 16.462248
11, 0.000000
12, 0.000000
13, 0.000000
14, 11.873317
15, 0.000000
16, 0.000000
17, 0.000000
18, 9.355519
19, 0.000000
20, 0.000000
21, 0.000000
22, 7.780546
23, 0.000000
24, 0.000000
25, 0.000000
26, 6.714797
27, 0.000000
28, 0.000000
29, 0.000000
30, 5.956290
31, 0.000000
32, 0.000000
33, 0.000000
34, 5.398467
35, 0.000000
36, 0.000000
37, 0.000000
38, 4.980033
39, 0.000000
40, 0.000000
41, 0.000000
42, 4.663480
43, 0.000000
44, 0.000000
45, 0.000000
46, 4.424831
47, 0.000000
48, 0.000000
49, 0.000000
50, 4.248341
51, 0.000000
52, 0.000000
53, 0.000000
54, 4.123578
55, 0.000000
56, 0.000000
57, 0.000000
58, 4.043768
59, 0.000000
60, 0.000000
61, 0.000000
62, 4.004824
63, 0.000000
64, 0.000000
65, 0.000000
66, 4.004824
67, 0.000000
68, 0.000000
69, 0.000000
70, 4.043768
71, 0.000000
72, 0.000000
73, 0.000000
74, 4.123578
75, 0.000000
76, 0.000000
77, 0